# Train L3 on Colab, deploy on your laptop
Commit/push the implementation branch first, then set `REPO_REF` below to that branch or commit. Select **Runtime → Change runtime type → GPU**. Colab only trains/exports: your application stays local. This notebook has not yet been GPU-executed. It makes no OpenRouter calls and needs no project API key. WANLI is CC BY 4.0; retain attribution in your presentation.
Use a new RUN name for each experiment. Drive checkpoints survive Colab runtime resets. Training/test splits and model revisions are frozen in manifests. Never place real candidate resumes in this notebook.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, time
REPO_URL = 'https://github.com/asinzuar/Agentic-Talent-Screening-Platform.git'
REPO_REF = "l3_nli"
REPO = Path('/content/ats-l3')
if REPO_REF.startswith('REPLACE_'): raise ValueError('Set REPO_REF to your pushed implementation branch or commit.')
if not REPO.exists(): subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_REF], check=True)
UV = shutil.which('uv')
if UV is None:
    # Bootstrap uv itself from PyPI; project dependencies still use uv.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
    UV = subprocess.check_output([sys.executable, '-c', 'import uv; print(uv.find_uv_bin())'], text=True).strip()
subprocess.run([UV, '--version'], check=True)
TRAIN_REQ = Path('/tmp/ats-l3-training-requirements.txt')
subprocess.run([UV, 'export', '--only-group', 'l3-train', '--no-emit-project', '--no-hashes', '--format', 'requirements.txt', '--output-file', str(TRAIN_REQ)], cwd=REPO, check=True)
training_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'))
def run_module(module, *args, extra=()):
    cmd = [UV, 'run', '--no-project', '--python', '3.12', '--with-requirements', str(TRAIN_REQ)]
    for package in extra: cmd += ['--with', package]
    log_dir = Path('/content/ats-l3-logs')
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / f'{module.rsplit(".", 1)[-1]}-{time.time_ns()}.log'
    print(f'Running {module}. Full output: {log_file}', flush=True)
    with log_file.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd + ['python', '-m', module, *map(str, args)], cwd=REPO,
                              env=dict(training_env, PYTHONUNBUFFERED='1'), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f'{module} failed (exit {returncode}). Read the actual error above; log: {log_file}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/ats-l3-nli')
DATA = ROOT / 'wanli-v1'
RUN = ROOT / 'qwen3b-run-001'
if not (DATA / 'manifest.json').exists():
    run_module('agentic_talent_screener.evaluation.nli', 'prepare', '--out', DATA)
print('Frozen data:', DATA)


## Train the adapter
Default: 3,000 train / 500 validation pairs, two epochs. Test labels are not loaded by training. Checkpoints are saved each epoch. Set RESUME to a saved checkpoint path after an interrupted session; keep the same data/config. Completion time and free GPU access are not guaranteed.

In [ ]:
from uuid import uuid4

RUN = ROOT / f"qwen3b-run-{uuid4().hex[:8]}"
RESUME = None

print("New run:", RUN)
run_module(
    "agentic_talent_screener.evaluation.nli_training",
    "train", "--data", DATA, "--out", RUN,
)

## Export for Ollama
This reloads the original FP16 base, merges the adapter and exports Q4_K_M GGUF. The original base is frozen. INCLUDE_BASELINE optionally exports an equivalently quantised unadapted base for a fair later comparison. The bundle includes the separate adapter and tokenizer. Intermediate FP16 files need additional Drive space. Conversion uses llama.cpp; its commit is recorded.
If Colab was reset after training, run the setup/Drive cells again before export.

In [ ]:
LLAMA = Path('/content/llama.cpp')
LLAMA_CPP_REF = None  # set a recorded commit to reproduce a previous export
if not LLAMA.exists(): subprocess.run(['git', 'clone', 'https://github.com/ggml-org/llama.cpp', str(LLAMA)], check=True)
if LLAMA_CPP_REF: subprocess.run(['git', '-C', str(LLAMA), 'checkout', LLAMA_CPP_REF], check=True)
subprocess.run(['cmake', '-S', str(LLAMA), '-B', str(LLAMA / 'build'), '-DGGML_CUDA=OFF', '-DLLAMA_CURL=OFF'], check=True)
subprocess.run(['cmake', '--build', str(LLAMA / 'build'), '--target', 'llama-quantize', '-j', '2'], check=True)
INCLUDE_BASELINE = False
args = ['export', '--run', RUN, '--llama-cpp', LLAMA]
if INCLUDE_BASELINE: args += ['--include-baseline']
run_module('agentic_talent_screener.evaluation.nli_training', *args, extra=('gguf', 'sentencepiece', 'protobuf'))
print('Download from Google Drive:', RUN / 'l3-model-bundle.zip')
print('Also download these frozen evaluation files:', DATA / 'test.jsonl', DATA / 'manifest.json')


## Back on Windows
Download the zip and test.jsonl from Drive. Extract to `artifacts/l3_nli/bundle` and put test.jsonl in `artifacts/l3_nli/data`. From the bundle folder run `ollama create ats-l3-qwen2.5-3b -f Modelfile.adapter`. Return to the repository and follow **docs/L3_NLI_EXPERIMENT.md** for the smoke test, dataset metrics, project claims suite and fixed-L2 replay. No training or model calls have run until you execute the relevant cells/commands.
The exported model can also run on EC2 with Ollama; save the adapter/base revision for other Hugging Face/SageMaker deployments.